In [ ]:
import warnings
warnings.simplefilter("ignore")
import numpy as np
import pandas as pd 
from lightkurve import LightCurve
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve 
import lightkurve as lk

for att in ['axes.labelsize', 'axes.titlesize', 'legend.fontsize',
            'legend.fontsize', 'xtick.labelsize', 'ytick.labelsize']:
    plt.rcParams[att] = 15

search_result = search_lightcurve("AU Mic") 
#print(search_result)
lc2min = search_result[4].download() 
lc2 = lc2min.remove_nans().remove_outliers()
lc2 = lc2[lc2.quality == 0]
lc2_m = lc2.normalize().remove_nans()

x2min = np.ascontiguousarray(lc2_m.time.value, dtype=np.float64) 
y2min = np.ascontiguousarray(lc2_m.flux, dtype=np.float64)
yerr2min = np.ascontiguousarray(lc2_m.flux_err, dtype=np.float64)
lcquality = np.ascontiguousarray(lc2_m.quality, dtype=np.float64)

: 

In [26]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt

tempos_click = []

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(lc2_m.time.value, lc2_m.flux, 'ko-', ms=2, lw=1, alpha=0.7, label='Original')
ax.set_title('Clique esquerdo: marca tempo | Backspace: desfaz último | Enter: finalizar')
ax.set_xlabel('Tempo [BTJD]')
ax.set_ylabel('Fluxo')

def onclick(event):
    if event.inaxes != ax or event.xdata is None:
        return
    # Registra apenas com duplo clique esquerdo
    if event.button == 1 and event.dblclick:
        tempos_click.append(float(event.xdata))
        ax.axvline(event.xdata, color='tab:red', ls='--', alpha=0.5)
        print(f'{len(tempos_click)} -> {event.xdata:.6f}')
        fig.canvas.draw_idle()

def onkey(event):
    if event.key == 'backspace' and len(tempos_click) > 0:
        removido = tempos_click.pop()
        print(f'Desfeito: {removido:.6f}')
        # Redesenha para atualizar linhas
        ax.clear()
        ax.plot(lc2_m.time.value, lc2_m.flux, 'ko-', ms=2, lw=1, alpha=0.7, label='Original')
        ax.set_title('Clique esquerdo: marca tempo | Backspace: desfaz último | Enter: finalizar')
        ax.set_xlabel('Tempo [BTJD]')
        ax.set_ylabel('Fluxo')
        for x in tempos_click:
            ax.axvline(x, color='tab:red', ls='--', alpha=0.5)
        fig.canvas.draw_idle()

    if event.key == 'enter':
        fig.canvas.mpl_disconnect(cid_click)
        fig.canvas.mpl_disconnect(cid_key)
        plt.close(fig)
        print('Captura finalizada.')
        print(f'Total de cliques: {len(tempos_click)}')

cid_click = fig.canvas.mpl_connect('button_press_event', onclick)
cid_key = fig.canvas.mpl_connect('key_press_event', onkey)

print('Instrucoes:')
print('- Botao esquerdo: adiciona tempo')
print('- Backspace: remove ultimo tempo')
print('- Enter: finaliza captura')
plt.show()

Instrucoes:
- Botao esquerdo: adiciona tempo
- Backspace: remove ultimo tempo
- Enter: finaliza captura


In [27]:
# Opcional: use esta celula se quiser encerrar captura manualmente
try:
    plt.gcf().canvas.mpl_disconnect(cid_click)
    plt.gcf().canvas.mpl_disconnect(cid_key)
    print('Captura desconectada manualmente.')
except Exception:
    print('Nada para desconectar (ou captura ja finalizada).')

Captura desconectada manualmente.


In [28]:
import numpy as np

if len(tempos_click) < 2:
    raise ValueError('Voce precisa de pelo menos 2 cliques.')

if len(tempos_click) % 2 != 0:
    print('Quantidade impar de cliques. O ultimo ponto sera ignorado.')

n = (len(tempos_click) // 2) * 2
pares = np.array(tempos_click[:n], dtype=float).reshape(-1, 2)

# Garante ordem inicio <= fim em cada par
pares = np.sort(pares, axis=1)

# Vetor pedido: mascara = [[ini1, fim1], [ini2, fim2], ...]
mascara = pares.tolist()

# Compatibilidade com o restante do pipeline atual
segmentos = [tuple(p) for p in pares]
mascara_flares_list = mascara

print('Vetor mascara [inicio, fim]:')
for i, (ini, fim) in enumerate(mascara, start=1):
    print(f'{i:02d}: [{ini:.6f}, {fim:.6f}]')

# Salva para reutilizar depois
np.savetxt(
    'intervalos_flares_transitos.csv',
    pares,
    delimiter=',',
    header='t_ini,t_fim',
    comments='',
    fmt='%.6f'
)
print('\nArquivo salvo: intervalos_flares_transitos.csv')

# Mantem backend interativo no notebook
%matplotlib widget

Vetor mascara [inicio, fim]:
01: [3883.239461, 3883.334868]
02: [3884.751868, 3884.843937]
03: [3885.024950, 3885.109542]
04: [3885.549959, 3885.617899]
05: [3885.738284, 3885.759441]
06: [3886.130183, 3886.411659]
07: [3888.081298, 3888.115210]
08: [3888.784350, 3888.805991]
09: [3889.179949, 3889.251286]
10: [3889.826802, 3889.854535]
11: [3890.321862, 3890.405131]
12: [3891.013277, 3891.076262]
13: [3891.741181, 3891.795356]
14: [3891.844004, 3891.908792]
15: [3891.950028, 3892.001562]
16: [3892.631057, 3892.697367]
17: [3892.812488, 3892.900441]
18: [3893.017865, 3893.101673]
19: [3893.403291, 3893.458281]
20: [3896.687407, 3896.849926]
21: [3897.567644, 3897.779646]
22: [3899.153300, 3899.218797]
23: [3899.542373, 3899.616046]
24: [3900.250646, 3900.422185]
25: [3900.680283, 3900.764591]
26: [3903.026735, 3903.311503]
27: [3904.512267, 3904.614076]
28: [3904.782061, 3904.847219]
29: [3905.340994, 3905.482584]

Arquivo salvo: intervalos_flares_transitos.csv


In [33]:
from astropy.stats import sigma_clip
import numpy as np

# ============================================================
# PASSO 1: SEGMENTAÇÃO E AJUSTE DE MANCHAS (ESTILO ARTIGO)
# ============================================================
t = np.array(lc2_m.time.value)
f = np.array(lc2_m.flux)

# --- AJUSTE ESTES PARÂMETROS PARA TESTAR ---
GRAU_POLINOMIO = 4     # Tente 4, 6 ou 8 se a curva for muito complexa
SIGMA_CLIPPING = 4.0   # Menor = mais rigoroso (limpa mais), Maior = segue mais a curva
DURACAO_MAX_SEG = 3.0  # Split automático de segmentos longos (em dias)

# 1. Detectar gaps do TESS (> 0.5 dias)
dt = np.diff(t)
gaps = t[np.where(dt > 0.5)[0]]

# 2. Criar limites iniciais (bins do TESS + splits de segurança)
limites = np.sort(np.unique(np.concatenate(([t.min(), t.max()], gaps))))

# 3. Subdividir segmentos muito longos (se um "pedaço" for > DURACAO_MAX_SEG)
limites_finais = [limites[0]]
for i in range(len(limites)-1):
    inicio, fim = limites[i], limites[i+1]
    while (fim - limites_finais[-1]) > DURACAO_MAX_SEG:
        limites_finais.append(limites_finais[-1] + DURACAO_MAX_SEG)
    limites_finais.append(fim)
limites_finais = np.unique(limites_finais)

segmentos = [(limites_finais[i], limites_finais[i+1]) for i in range(len(limites_finais)-1)]

print(f"Segmentação: {len(segmentos)} trechos independentes.")

# --- INICIANDO AJUSTE ---
modelo_manchas = np.zeros_like(f)

for i, (seg_ini, seg_fim) in enumerate(segmentos):
    idx = (t >= seg_ini) & (t <= seg_fim)
    if not np.any(idx): continue
    
    t_seg = t[idx]
    f_seg = f[idx]
    
    # Centralizar o tempo para evitar RankWarning (problema numérico de escala)
    t_mid = np.median(t_seg)
    t_shifted = t_seg - t_mid
    
    good = np.ones(len(t_seg), dtype=bool)
    
    # Ajuste Iterativo (Conforme o Artigo: Polinômio + Sigma Clipping)
    for _ in range(5):
        if np.sum(good) < (GRAU_POLINOMIO + 2): break
        
        # Ajuste com tempo centralizado
        coef = np.polyfit(t_shifted[good], f_seg[good], deg=GRAU_POLINOMIO)
        modelo_local = np.polyval(coef, t_shifted)
        
        # Clipping
        resid = f_seg - modelo_local
        clipped = sigma_clip(resid, sigma=SIGMA_CLIPPING, maxiters=1)
        
        if clipped.mask is np.ma.nomask:
            break # Convergiu o clipping
            
        good = (~clipped.mask) & good
        
    modelo_manchas[idx] = modelo_local

# Residual para inspeção
residual_manchas = f / modelo_manchas
print("✓ Ajuste de manchas concluído. Prossiga para o gráfico de teste.")

Segmentação: 9 trechos independentes.
✓ Ajuste de manchas concluído. Prossiga para o gráfico de teste.


In [35]:

# ============================================================
# PASSO 3: VISUALIZAR RESIDUAIS E SELECIONAR FLARES INTERATIVAMENTE
# ============================================================
print("\n" + "="*60)
print("VISUALIZAÇÃO: Residuais após manchas (para seleção de flares)")
print("="*60)

%matplotlib qt
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Gráfico 1: Dados originais + manchas
ax1.plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.6, label='Dados Originais')
ax1.plot(t, modelo_manchas, 'r-', linewidth=2.5, label='Ajuste (Manchas)')
ax1.set_ylabel('Fluxo Normalizado', fontsize=14)
ax1.set_title('Passo 1: Ajuste de Manchas Estelares', fontsize=15, fontweight='bold')
ax1.legend(loc='upper right', fontsize=12)
ax1.grid(alpha=0.3)

# Gráfico 2: Residuais para detecção de flares
ax2.plot(t, residual_manchas, 'b.-', ms=1.5, lw=0.5, alpha=0.7, label='Residual (original/manchas)')
ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax2.set_ylabel('Fluxo Residual', fontsize=14)
ax2.set_xlabel('Tempo [BTJD dias]', fontsize=14)
ax2.set_title('Residuais para Seleção de Flares e Trânsitos', fontsize=15, fontweight='bold')
ax2.legend(loc='upper right', fontsize=12)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n→ Feche o gráfico e continue na próxima célula para inserir os intervalos de flares/trânsitos")



VISUALIZAÇÃO: Residuais após manchas (para seleção de flares)

→ Feche o gráfico e continue na próxima célula para inserir os intervalos de flares/trânsitos


In [ ]:

# ============================================================
# PASSO 4: ENTRADA INTERATIVA DE FLARES E TRÂNSITOS
# ============================================================
print("\n" + "="*60)
print("ENTRADA: Defina os intervalos de FLARES e TRÂNSITOS")
print("="*60)

print("""
Formato: Digite os intervalos como uma lista de pares [inicio, fim]

Exemplo:
[1326.67, 1326.78], [1327.45, 1327.56], [1328.12, 1328.23]

Copie e cole os intervalos que você quer remover do ajuste:
""")

# Entrada do usuário
entrada_flares_str = input("Cole os intervalos de flares/trânsitos (ou Enter para nenhum): ").strip()

if entrada_flares_str:
    try:
        # Converte string em lista de listas
        mascara_flares_list = eval("[" + entrada_flares_str + "]")
        
        # Validação
        if isinstance(mascara_flares_list[0], (int, float)):
            # Caso seja um único intervalo [ini, fim]
            mascara_flares_list = [mascara_flares_list]
        
        print(f"\n✓ {len(mascara_flares_list)} intervalo(s) detectado(s):")
        for i, (ini, fim) in enumerate(mascara_flares_list):
            print(f"  {i+1}. [{ini:.4f}, {fim:.4f}]")
    except:
        print("✗ Erro ao processar entrada. Usando lista vazia.")
        mascara_flares_list = []
else:
    mascara_flares_list = []
    print("→ Nenhum flare/trânsito definido, usando toda a curva")

# Criar máscara booleana
mascara_flares = np.zeros(len(t), dtype=bool)
for ini, fim in mascara_flares_list:
    mascara_flares |= (t >= ini) & (t <= fim)

mask_good_flares = ~mascara_flares

print(f"\nMáscara criada:")
print(f"  Pontos excluídos (flares/trânsitos): {mascara_flares.sum()}")
print(f"  Pontos para ajuste: {mask_good_flares.sum()}")



ENTRADA: Defina os intervalos de FLARES e TRÂNSITOS

Formato: Digite os intervalos como uma lista de pares [inicio, fim]

Exemplo:
[1326.67, 1326.78], [1327.45, 1327.56], [1328.12, 1328.23]

Copie e cole os intervalos que você quer remover do ajuste:



In [ ]:

# ============================================================
# PASSO 5: SEGUNDA ITERAÇÃO - AJUSTE DE FLARES/TRÂNSITOS
# COM CONVERGÊNCIA (MELHORA RMS < 1%)
# ============================================================
print("\n" + "="*60)
print("SEGUNDA ITERAÇÃO: Ajuste de Flares/Trânsitos com Convergência")
print("="*60)

# Começa com modelo de manchas como baseline
modelo_flares = modelo_manchas.copy()
residual_anterior = residual_manchas.copy()
rms_inicial = np.sqrt(np.mean((residual_anterior - 1.0)**2))

convergencia_atingida = False
num_iteracoes_globais = 0
MAX_ITERACOES = 10  # Limite de segurança

while not convergencia_atingida and num_iteracoes_globais < MAX_ITERACOES:
    num_iteracoes_globais += 1
    print(f"\n--- Iteração Global {num_iteracoes_globais} ---")
    
    # Ajusta flares/trânsitos excluindo os intervalos definidos
    modelo_flares_novo = np.zeros_like(f)
    
    for seg_num, (seg_ini, seg_fim) in enumerate(segmentos):
        seg = (t >= seg_ini) & (t <= seg_fim)
        if not np.any(seg): continue
        
        t_seg = t[seg]
        f_seg = f[seg]
        
        # Centralizar tempo para evitar RankWarning
        t_mid = np.median(t_seg)
        t_shifted = t_seg - t_mid
        
        good = mask_good_flares[seg].copy()
        
        for iter_num in range(5):  # 5 iterações por segmento
            if good.sum() < (4 + 2): # Conferir se tem pontos para deg=4
                break
            
            coef = np.polyfit(t_shifted[good], f_seg[good], deg=4)
            modelo = np.polyval(coef, t_shifted)
            resid = f_seg - modelo
            
            clipped = sigma_clip(resid, sigma=2.5, maxiters=1)
            
            if clipped.mask is np.ma.nomask:
                new_good = good.copy()
            else:
                new_good = (~clipped.mask) & good
            
            good = new_good
        
        modelo_flares_novo[seg] = modelo
    
    # Calcula novo residual
    residual_novo = f / modelo_flares_novo
    rms_novo = np.sqrt(np.mean((residual_novo - 1.0)**2))
    
    # RMS anterior calculado apenas nos pontos bons para comparação justa
    rms_ant_check = np.sqrt(np.mean((residual_anterior - 1.0)**2))
    melhora_relativa = (1 - rms_novo / rms_ant_check) * 100
    
    print(f"  RMS anterior: {rms_ant_check:.6f}")
    print(f"  RMS novo:     {rms_novo:.6f}")
    print(f"  Melhora:      {melhora_relativa:.2f}%")
    
    if melhora_relativa < 1.0:
        print(f"\n✓ CONVERGÊNCIA ATINGIDA! (melhora < 1%)")
        convergencia_atingida = True
    else:
        modelo_flares = modelo_flares_novo.copy()
        residual_anterior = residual_novo.copy()

# Resultado final
tempo_trend = t
fluxo_trend = modelo_flares
fluxo_flat = f / modelo_flares
tempo_flat = t

print(f"\n" + "="*60)
print(f"✓ Detrending concluído!")
print(f"  Total de iterações globais: {num_iteracoes_globais}")
print(f"  RMS final: {np.sqrt(np.mean((residual_novo - 1.0)**2)):.6f}")
print(f"="*60)


In [ ]:
%matplotlib qt
import matplotlib.pyplot as plt

print("\n" + "="*60)
print("GRÁFICOS FINAIS: Comparação Original vs Detrended")
print("="*60)

fig, axes = plt.subplots(3, 1, figsize=(18, 12))

# --- Gráfico 1: Dados originais + Ajuste final ---
axes[0].plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.6, label='Dados Originais')
axes[0].plot(tempo_trend, fluxo_trend, 'r-', linewidth=2.5, label='Ajuste Final (Manchas+Flares)')
axes[0].set_ylabel('Fluxo Normalizado', fontsize=14)
axes[0].set_title('Passo 1: Dados Originais + Tendência', fontsize=15, fontweight='bold')
axes[0].legend(loc='upper right', fontsize=12)
axes[0].grid(alpha=0.3)

# --- Gráfico 2: Residual detrended ---
axes[1].plot(tempo_flat, fluxo_flat, 'b.-', ms=1.5, lw=0.5, alpha=0.7)
axes[1].axhline(y=1.0, color='gray', linestyle='--', linewidth=1.5, alpha=0.7)
axes[1].fill_between(tempo_flat, 0.98, 1.02, alpha=0.2, color='green', label='Band ±2%')
axes[1].set_ylabel('Fluxo Normalizado', fontsize=14)
axes[1].set_title('Passo 2: Curva Detrended (Residuais)', fontsize=15, fontweight='bold')
axes[1].legend(loc='upper right', fontsize=12)
axes[1].grid(alpha=0.3)

# --- Gráfico 3: Zoom em região com flares ---
if len(mascara_flares_list) > 0:
    zoom_ini, zoom_fim = mascara_flares_list[0]
    zoom_mask = (t >= zoom_ini - 0.05) & (t <= zoom_fim + 0.05)
    
    axes[2].plot(t[zoom_mask], f[zoom_mask], 'ko-', ms=3, lw=1, alpha=0.7, label='Original')
    axes[2].plot(tempo_trend[zoom_mask], fluxo_trend[zoom_mask], 'r-', linewidth=2.5, label='Ajuste')
    
    # Marca a região do flare
    for ini, fim in mascara_flares_list:
        axes[2].axvspan(ini, fim, alpha=0.2, color='red', label='Flare/Trânsito (excluído)' if ini == mascara_flares_list[0][0] else '')
    
    axes[2].set_xlabel('Tempo [BTJD dias]', fontsize=14)
    axes[2].set_ylabel('Fluxo Normalizado', fontsize=14)
    axes[2].set_title('Zoom: Região com Flare/Trânsito', fontsize=15, fontweight='bold')
    axes[2].legend(loc='upper right', fontsize=12)
    axes[2].grid(alpha=0.3)
else:
    axes[2].text(0.5, 0.5, 'Nenhum flare/trânsito selecionado', 
                 ha='center', va='center', transform=axes[2].transAxes, fontsize=14)

plt.tight_layout()
plt.show()

print("\n✓ Gráficos plotados com sucesso!")


Variáveis disponíveis:
  t shape: (15111,)
  f shape: (15111,)
  tempo_trend shape: (15111,)
  fluxo_trend shape: (15111,)
Gráfico plotado com sucesso!
